# NLP Analysis of AI Disclosures in 10-K Filings with Gemini

A structured NLP pipeline for analysing AI-related disclosures in long-form corporate filings.

> **Academic context.** The work originated in graduate coursework for *Applied Machine Learning in Finance* at the University of Melbourne and has been adapted for public presentation.

## Project Overview

This project examines how ten major US-listed companies describe artificial intelligence in their Form 10-K filings. It combines deterministic document preprocessing with Gemini-based information extraction to transform unstructured filing text into consistent company-level representations.

The extraction schema covers AI-related risk, opportunity, investment disclosure, and the latest explicit investment value in billions of US dollars. These representations also support a constrained Bullish, Bearish, or Neutral classification of each company's AI positioning.

The public portfolio version emphasises NLP research design, structured extraction, and model limitations rather than a single set of generated results.

In [ ]:
from pathlib import Path
import os
import re

from bs4 import BeautifulSoup
import google.generativeai as genai
import pandas as pd
import tqdm

## Data and Document Corpus

The original analysis used ten company 10-K HTML files supplied through the University of Melbourne course environment. The raw files are not included in this repository.

To reproduce the analysis with independently obtained public filings, place authorised copies in `data/` using the pattern `<TICKER>_latest_10K.html`. Filing dates and contents may differ from the original corpus, so new outputs may not exactly match the original run.

In [ ]:
DATA_DIR = Path("data")
TICKERS = ["NVDA", "GOOGL", "MSFT", "AMZN", "META",
           "TSLA", "IBM", "INTC", "CRM", "ORCL"]

filing_paths = {ticker: DATA_DIR / f"{ticker}_latest_10K.html" for ticker in TICKERS}
missing_files = [str(path) for path in filing_paths.values() if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "Raw filings are not included. Add authorised copies to data/. "
        f"Missing {len(missing_files)} file(s)."
    )

## Document Preprocessing

Each filing is read as HTML and converted to a normalised plain-text representation. Beautiful Soup removes markup, while regular expressions remove URLs and collapse repeated whitespace. This creates a consistent textual input without embedding raw filings in the notebook.

In [ ]:
def load_html_file(file_path):
    """Read a complete HTML filing as a UTF-8 string."""
    with open(file_path, "r", encoding="utf-8") as file:
        return file.read()


def remove_html_tags(html_content):
    """Convert filing HTML to normalised plain text."""
    soup = BeautifulSoup(html_content, "html.parser")
    plain_text = soup.get_text(" ")
    plain_text = re.sub(r"http\S+", "", plain_text)
    return re.sub(r"\s+", " ", plain_text).strip()

## Model Configuration

The original implementation used Gemini 2.0 Flash. The API key is loaded from an environment variable and is never stored in the notebook or repository.

In [ ]:
GOOGLE_GEMINI_API_KEY = os.environ["GOOGLE_GEMINI_API_KEY"]
genai.configure(api_key=GOOGLE_GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.0-flash")

## Prompt and Extraction Schema

The prompt translates a qualitative research objective into a predictable response format. It defines the semantic categories, distinguishes planned from completed investment, specifies the numeric unit, and instructs the model to return `null` when the filing lacks an explicit value.

In [ ]:
def build_extraction_prompt(filing_text):
    return f"""
You are a financial analyst specialising in company 10-K reports.

Based only on the filing text, summarise the company's view of:
- AI-related Risk
- AI-related Opportunity
- AI-related Investment

For investment:
- Report planned or committed AI-related investment only.
- Do not combine completed and planned investment.
- Express an explicit amount in billions of US dollars.
- Return null when the filing does not disclose a specific amount.
- Do not infer unsupported information.

Use exactly this structure:
**AI-related Risk:**
<2-3 concise sentences>

**AI-related Opportunity:**
<2-3 concise sentences>

**AI-related Investment:**
<2-3 concise sentences>
Investment Value (Billion USD): <number or null>
Investment Year (Latest year): <YYYY or null>

Filing text:
{filing_text}
"""

## Structured Response Parsing

The generated response is mapped into a fixed pandas Series. Qualitative sections are split using the requested headings, while a regular expression extracts the numeric investment field. This combines probabilistic generation with deterministic post-processing.

In [ ]:
def parse_sections(response_text):
    risk = response_text.split("**AI-related Risk:**")[-1]\
        .split("**AI-related Opportunity:**")[0].strip()
    opportunity = response_text.split("**AI-related Opportunity:**")[-1]\
        .split("**AI-related Investment:**")[0].strip()
    investment = response_text.split("**AI-related Investment:**")[-1]\
        .split("Investment Value (Billion USD):")[0].strip()

    value_match = re.search(
        r"Investment Value \(Billion USD\):\s*([0-9]*\.?[0-9]+|null)",
        response_text,
        flags=re.IGNORECASE,
    )

    if value_match and value_match.group(1).lower() != "null":
        investment_value = float(value_match.group(1))
    else:
        investment_value = None

    return pd.Series({
        "Risk": risk,
        "Opportunity": opportunity,
        "Investment": investment,
        "Investment Value (Billion USD)": investment_value,
    })

## Single-Filing Extraction

The preprocessing, prompting, inference, and parsing stages are wrapped in one reusable function. This separates the analytical interface from company-specific file handling.

In [ ]:
def extract_filing(ticker, model):
    html_content = load_html_file(filing_paths[ticker])
    filing_text = remove_html_tags(html_content)
    prompt = build_extraction_prompt(filing_text)
    response_text = model.generate_content(prompt).text
    return parse_sections(response_text)

## Corpus-Level Information Extraction

The same extraction function is applied across the ten-filing corpus. Results are aggregated into a company-indexed DataFrame, producing a structured representation suitable for comparison or downstream analysis.

In [ ]:
records = []

for ticker in tqdm.tqdm(TICKERS, desc="Extracting filing information"):
    extracted = extract_filing(ticker, model)
    extracted.name = ticker
    records.append(extracted)

extracted_df = pd.DataFrame(records)
extracted_df.index.name = "Ticker"
extracted_df.to_csv("10k_extracted_info.csv")
extracted_df

## Downstream AI Sentiment Classification

The extracted representations are assembled into a second prompt that constrains the model to one of three labels: Bullish, Bearish, or Neutral. This stage illustrates how structured document representations can support a downstream qualitative classification task.

In [ ]:
def format_company_information(ticker, row):
    value = row["Investment Value (Billion USD)"]
    value_text = "null" if pd.isna(value) else str(value)
    return (
        f"{ticker}:\n"
        f"- Risk: {row['Risk']}\n"
        f"- Opportunity: {row['Opportunity']}\n"
        f"- Investment: {row['Investment']}\n"
        f"- Investment Value (Billion USD): {value_text}\n"
    )


company_information = "\n".join(
    format_company_information(ticker, extracted_df.loc[ticker])
    for ticker in extracted_df.index
)

ranking_prompt = f"""
Based only on the extracted information below, classify every company as
Bullish, Bearish, or Neutral on AI and provide a one-sentence rationale.
Do not introduce outside information.

Return one company per line as: Ticker: Sentiment - rationale.

{company_information}
"""

sentiment_text = model.generate_content(ranking_prompt).text
print(sentiment_text)

## Pipeline Demonstration

The original run generated structured company-level representations for all ten filings, demonstrating that a consistent extraction schema could be applied across a heterogeneous document corpus.

The outputs are illustrative rather than definitive empirical findings. The portfolio's main contribution is the end-to-end NLP design: document normalisation, task specification, LLM-assisted extraction, deterministic parsing, and corpus-level aggregation.

## Limitations

- Full 10-K filings may exceed practical model context limits, potentially causing relevant information to be omitted.
- LLM-generated extractions may vary across runs or include claims that are not fully supported by the source text.
- The extraction framework was evaluated using a single model and was not benchmarked against alternative LLMs, limiting conclusions about cross-model robustness.
- AI investment disclosures are not standardised across companies, limiting the comparability of extracted values.
- The outputs are exploratory and should be verified against the original filings before interpretation.

## Responsible Use

This notebook is an educational portfolio artifact, not investment advice. Model-generated claims should be verified against the original SEC filing before interpretation. The underlying course dataset and teaching materials are not distributed in this repository.